In [0]:
table_name = "fact_labs"
silver_table_fqn = f"he_prod_incen_analytics_dbw_01.silver.{table_name}"

df = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.fact_labs")

In [0]:
from pyspark.sql.functions import col, upper, trim, when, to_date, row_number, abs as spark_abs
from pyspark.sql.window import Window

# 1. Enrich: Map Source SKs -> Business Keys
df_map_member = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_members").select(col("member_sk").cast("int").alias("src_member_sk"), col("member_id"))
df_map_provider = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_providers").select(col("provider_sk").cast("int").alias("src_provider_sk"), col("provider_id"))
df_map_facility = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_facilities").select(col("facility_sk").cast("int").alias("src_facility_sk"), col("facility_id"))
df_map_cpt = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_cpt_codes").select(col("cpt_code_sk").cast("int").alias("src_cpt_code_sk"), col("cpt_code"))

df = df.join(df_map_member, df["member_sk"] == df_map_member["src_member_sk"], "left").drop("src_member_sk", "member_sk")
df = df.join(df_map_provider, df["provider_sk"] == df_map_provider["src_provider_sk"], "left").drop("src_provider_sk", "provider_sk")
df = df.join(df_map_facility, df["facility_sk"] == df_map_facility["src_facility_sk"], "left").drop("src_facility_sk", "facility_sk")
df = df.join(df_map_cpt, df["cpt_code_sk"] == df_map_cpt["src_cpt_code_sk"], "left").drop("src_cpt_code_sk", "cpt_code_sk")

# 2. Standardize, Cast Dates/Ints/Decimals/Booleans
for c in ["lab_order_id", "member_id", "provider_id", "facility_id", "cpt_code", "lab_type", "test_name", "loinc_code", "unit_of_measure"]:
    df = df.withColumn(c, upper(trim(col(c))))

for date_col in ["order_date", "collection_date", "result_date"]:
    df = df.withColumn(date_col, to_date(trim(col(date_col))))

df = df.withColumn("turnaround_hours", trim(col("turnaround_hours")).cast("int"))

for dec_col in ["test_value", "reference_range_low", "reference_range_high", "lab_charge", "insurance_paid", "member_responsibility"]:
    is_valid = trim(col(dec_col)).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
    df = df.withColumn(dec_col, when(is_valid, trim(col(dec_col)).cast("decimal(15,2)")).otherwise(None))
    if dec_col in ["lab_charge", "insurance_paid", "member_responsibility"]:
        df = df.withColumn(dec_col, spark_abs(col(dec_col)))

for bool_col in ["abnormal_flag", "critical_value", "specialty_lab", "quality_flag"]:
    df = df.withColumn(bool_col, when(upper(trim(col(bool_col))).isin("YES", "1", "TRUE", "ABNORMAL", "CRITICAL"), True).when(upper(trim(col(bool_col))).isin("NO", "0", "FALSE", "NORMAL"), False).otherwise(None).cast("boolean"))

dedup_window = Window.partitionBy("lab_order_id").orderBy(col("_ingested_at").desc())
df = df.withColumn("_dq_is_deduped", row_number().over(dedup_window)).filter(col("_dq_is_deduped") == 1).drop("_dq_is_deduped", "_ingested_at", "_source_file", "lab_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table_fqn)
print(f"✅ Successfully wrote {table_name} to Silver layer.")